# TripPulse — Week 08: Gold Hand-off and Power BI Foundation

**Scope:** Batch Gold hand-off + Power BI foundation.  
**Week 10 streaming is NOT part of this notebook.**

## Week 08 objective
Create small, deterministic, student-owned Gold exports; validate them against the governed Gold tables; and prepare the source/traceability evidence for **PBI-01 — Ride Operations Overview**.

**Approved Gold sources:** `workspace.default` Gold tables only.

Power BI must not connect to raw/source, Bronze, Silver Candidate, Trusted Silver detail, quarantine, or technical-validation reference outputs.


In [0]:
# WEEK 08 — Cell 1: Catalog and approved Gold table inventory

CATALOG = "workspace"
SCHEMA = "default"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

gold_tables = [
    "dim_date",
    "dim_zone",
    "dim_driver",
    "fact_trip",
    "fact_payment_attempt",
    "agg_trip_operations_daily",
    "agg_zone_demand_daily",
    "agg_driver_performance_daily",
    "agg_surge_impact_daily",
    "agg_payment_reliability_daily"
]

existing_tables = {
    r.tableName
    for r in spark.sql("SHOW TABLES").collect()
}

inventory = [(t, t in existing_tables) for t in gold_tables]

display(
    spark.createDataFrame(
        inventory,
        ["gold_table", "exists_in_workspace_default"]
    )
)


gold_table,exists_in_workspace_default
dim_date,true
dim_zone,true
dim_driver,true
fact_trip,true
fact_payment_attempt,true
agg_trip_operations_daily,true
agg_zone_demand_daily,true
agg_driver_performance_daily,true
agg_surge_impact_daily,true
agg_payment_reliability_daily,true


## 2. Inspect the actual Gold schemas

Run this first. The actual Week 07 Gold schemas are the source of truth for the Week 08 export validation.


In [0]:
# WEEK 08 — Cell 2: Gold schema inspection

for table_name in gold_tables:
    print(f"\n===== {CATALOG}.{SCHEMA}.{table_name} =====")
    spark.table(table_name).printSchema()



===== workspace.default.dim_date =====
root
 |-- date_key: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- week_of_year: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)


===== workspace.default.dim_zone =====
root
 |-- zone_id: string (nullable = true)
 |-- zone_name: string (nullable = true)
 |-- zone_type: string (nullable = true)
 |-- city_code: string (nullable = true)
 |-- demand_band: string (nullable = true)
 |-- is_active: boolean (nullable = true)


===== workspace.default.dim_driver =====
root
 |-- driver_id: string (nullable = true)
 |-- home_zone_id: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- service_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- rating: decimal(3,2) (nullable = t

In [0]:
# WEEK 08 — Cell 3: Gold row counts and column counts

count_rows = []

for table_name in gold_tables:
    df = spark.table(table_name)
    count_rows.append(
        (table_name, df.count(), len(df.columns))
    )

display(
    spark.createDataFrame(
        count_rows,
        ["gold_table", "row_count", "column_count"]
    )
)


gold_table,row_count,column_count
dim_date,91,9
dim_zone,120,6
dim_driver,2793,7
fact_trip,241654,30
fact_payment_attempt,172403,19
agg_trip_operations_daily,360,12
agg_zone_demand_daily,42945,8
agg_driver_performance_daily,140619,9
agg_surge_impact_daily,80116,10
agg_payment_reliability_daily,340,9


## 3. Week 08 export preparation register

The approved manual identifies these export families:

- `dim_date`, `dim_zone`, `dim_driver` — relationships, labels and slicers
- `agg_trip_operations_daily` — PBI-01 cards and trends
- `agg_zone_demand_daily`, `agg_surge_impact_daily` — zone detail and PBI-02
- `agg_driver_performance_daily`, `agg_payment_reliability_daily` — PBI-03

This week implements **PBI-01 only**, but the hand-off notebook prepares the approved Gold export families needed across the three-page report.


In [0]:
# WEEK 08 — Cell 4: Build the deterministic 14-day Gold slice

from pyspark.sql import functions as F

trip_ops = spark.table("agg_trip_operations_daily")

# Actual Week 07 Gold schema uses operation_date for daily summaries.
date_col = "operation_date"

if date_col not in trip_ops.columns:
    raise ValueError(
        "agg_trip_operations_daily does not contain operation_date. "
        "Check the actual Week 07 Gold schema before continuing."
    )

date_slice = (
    trip_ops
    .select(date_col)
    .where(F.col(date_col).isNotNull())
    .distinct()
    .orderBy(F.col(date_col))
    .limit(14)
)

date_values = [r[date_col] for r in date_slice.collect()]

print("Date column:", date_col)
print("Deterministic date slice:", date_values)
print("Number of dates:", len(date_values))

if not date_values:
    raise ValueError("No non-null operation_date values were found in agg_trip_operations_daily.")


Date column: operation_date
Deterministic date slice: [datetime.date(2026, 1, 1), datetime.date(2026, 1, 2), datetime.date(2026, 1, 3), datetime.date(2026, 1, 4), datetime.date(2026, 1, 5), datetime.date(2026, 1, 6), datetime.date(2026, 1, 7), datetime.date(2026, 1, 8), datetime.date(2026, 1, 9), datetime.date(2026, 1, 10), datetime.date(2026, 1, 11), datetime.date(2026, 1, 12), datetime.date(2026, 1, 13), datetime.date(2026, 1, 14)]
Number of dates: 14


In [0]:
# WEEK 08 — Cell 5: Validate that all required summary tables support the same date slice

summary_tables = [
    "agg_trip_operations_daily",
    "agg_zone_demand_daily",
    "agg_surge_impact_daily",
    "agg_driver_performance_daily",
    "agg_payment_reliability_daily"
]

# The first four daily summaries use operation_date; payment reliability uses payment_date.
summary_date_columns = {
    "agg_trip_operations_daily": "operation_date",
    "agg_zone_demand_daily": "operation_date",
    "agg_surge_impact_daily": "operation_date",
    "agg_driver_performance_daily": "operation_date",
    "agg_payment_reliability_daily": "payment_date"
}

summary_slice_check = []

for table_name in summary_tables:
    df = spark.table(table_name)
    table_date_col = summary_date_columns[table_name]

    if table_date_col not in df.columns:
        summary_slice_check.append(
            (table_name, "FAIL", "missing_date_column", 0)
        )
    else:
        sliced = df.where(F.col(table_date_col).isin(date_values))
        summary_slice_check.append(
            (table_name, "PASS", table_date_col, sliced.count())
        )

display(
    spark.createDataFrame(
        summary_slice_check,
        ["gold_table", "status", "date_column", "slice_rows"]
    )
)


gold_table,status,date_column,slice_rows
agg_trip_operations_daily,PASS,operation_date,56
agg_zone_demand_daily,PASS,operation_date,6678
agg_surge_impact_daily,PASS,operation_date,12476
agg_driver_performance_daily,PASS,operation_date,21954
agg_payment_reliability_daily,PASS,payment_date,36


## 4. Create the deterministic student-owned export DataFrames

The notebook uses:

- first 90 ordered `dim_date` rows;
- all governed `dim_zone` rows;
- first 500 ordered `dim_driver` rows;
- first 14 ordered date keys for the five daily summary tables.

The exported values are never manually edited.


In [0]:
# WEEK 08 — Cell 6: Create all approved Week 08 export families

exports = {}

# ------------------------------------------------------------
# DIMENSIONS — used across PBI-01, PBI-02 and PBI-03
# ------------------------------------------------------------

dim_date = spark.table("dim_date")
if "date_key" in dim_date.columns:
    dim_date = dim_date.orderBy("date_key")
exports["dim_date"] = dim_date.limit(90)

dim_zone = spark.table("dim_zone")
zone_order = [c for c in ["zone_id", "zone_key"] if c in dim_zone.columns]
if zone_order:
    dim_zone = dim_zone.orderBy(*zone_order)
exports["dim_zone"] = dim_zone

dim_driver = spark.table("dim_driver")
driver_order = [c for c in ["driver_id", "driver_key"] if c in dim_driver.columns]
if driver_order:
    dim_driver = dim_driver.orderBy(*driver_order)
exports["dim_driver"] = dim_driver.limit(500)

# ------------------------------------------------------------
# DAILY GOLD SUMMARIES
# ------------------------------------------------------------

summary_date_columns = {
    "agg_trip_operations_daily": "operation_date",
    "agg_zone_demand_daily": "operation_date",
    "agg_surge_impact_daily": "operation_date",
    "agg_driver_performance_daily": "operation_date",
    "agg_payment_reliability_daily": "payment_date"
}

for table_name in summary_tables:
    df = spark.table(table_name)
    table_date_col = summary_date_columns[table_name]

    if table_date_col not in df.columns:
        raise ValueError(
            f"{table_name} does not contain {table_date_col}. Stop before export."
        )

    exports[table_name] = df.where(F.col(table_date_col).isin(date_values))

# Confirm every approved export family is present.
approved_export_families = [
    "dim_date",
    "dim_zone",
    "dim_driver",
    "agg_trip_operations_daily",
    "agg_zone_demand_daily",
    "agg_surge_impact_daily",
    "agg_driver_performance_daily",
    "agg_payment_reliability_daily"
]

missing_exports = [
    t for t in approved_export_families
    if t not in exports
]

if missing_exports:
    raise ValueError(
        f"Missing approved export families: {missing_exports}"
    )

for name, df in exports.items():
    print(f"{name}: rows={df.count()}, columns={len(df.columns)}")


dim_date: rows=90, columns=9
dim_zone: rows=120, columns=6
dim_driver: rows=500, columns=7
agg_trip_operations_daily: rows=56, columns=12
agg_zone_demand_daily: rows=6678, columns=8
agg_surge_impact_daily: rows=12476, columns=10
agg_driver_performance_daily: rows=21954, columns=9
agg_payment_reliability_daily: rows=36, columns=9


In [0]:
# WEEK 08 — Cell 7: Validate source schema and export DataFrame schema

schema_checks = []

for name, export_df in exports.items():
    source_df = spark.table(name)

    columns_match = source_df.columns == export_df.columns
    types_match = source_df.schema == export_df.schema

    schema_checks.append(
        (
            name,
            columns_match,
            types_match,
            len(source_df.columns),
            len(export_df.columns)
        )
    )

schema_check_df = spark.createDataFrame(
    schema_checks,
    [
        "gold_table",
        "columns_match_source",
        "types_match_source",
        "source_column_count",
        "export_column_count"
    ]
)

display(schema_check_df)

if not all(r["columns_match_source"] and r["types_match_source"]
           for r in schema_check_df.collect()):
    raise ValueError(
        "Schema/type validation failed. Do not export until corrected upstream."
    )


gold_table,columns_match_source,types_match_source,source_column_count,export_column_count
dim_date,true,true,9,9
dim_zone,true,true,6,6
dim_driver,true,true,7,7
agg_trip_operations_daily,true,true,12,12
agg_zone_demand_daily,true,true,8,8
agg_surge_impact_daily,true,true,10,10
agg_driver_performance_daily,true,true,9,9
agg_payment_reliability_daily,true,true,9,9


## 5. Key and grain validation

For Week 08, the important proof is that the export preserves the declared Gold grain.

The checks below use the approved business keys where those columns exist. For summary tables, the notebook reports duplicate groups for the expected grain columns rather than changing the Gold data.


In [0]:
# WEEK 08 — Cell 8: Key/grain checks

# Grain is based on the actual Week 07 Gold schemas.
grain_candidates = {
    "dim_date": ["date_key"],
    "dim_zone": ["zone_id"],
    "dim_driver": ["driver_id"],
    "agg_trip_operations_daily": ["operation_date", "service_type"],
    "agg_zone_demand_daily": ["operation_date", "pickup_zone_id", "service_type"],
    "agg_driver_performance_daily": ["operation_date", "driver_id"],
    "agg_surge_impact_daily": ["operation_date", "pickup_zone_id", "service_type", "surge_band"],
    "agg_payment_reliability_daily": ["payment_date", "payment_method"]
}

grain_results = []

for name, df in exports.items():
    candidates = grain_candidates.get(name, [])
    key_cols = [c for c in candidates if c in df.columns]

    if key_cols:
        duplicate_groups = (
            df.groupBy(*key_cols)
              .count()
              .where(F.col("count") > 1)
              .count()
        )
        grain_status = "PASS" if duplicate_groups == 0 else "CHECK"
    else:
        duplicate_groups = None
        grain_status = "NOT_CHECKED"

    grain_results.append(
        (name, ",".join(key_cols), duplicate_groups, grain_status)
    )

display(
    spark.createDataFrame(
        grain_results,
        ["gold_table", "checked_grain_columns", "duplicate_groups", "grain_status"]
    )
)


gold_table,checked_grain_columns,duplicate_groups,grain_status
dim_date,date_key,0,PASS
dim_zone,zone_id,0,PASS
dim_driver,driver_id,0,PASS
agg_trip_operations_daily,"operation_date,service_type",0,PASS
agg_zone_demand_daily,"operation_date,pickup_zone_id,service_type",0,PASS
agg_surge_impact_daily,"operation_date,pickup_zone_id,service_type,surge_band",0,PASS
agg_driver_performance_daily,"operation_date,driver_id",0,PASS
agg_payment_reliability_daily,"payment_date,payment_method",0,PASS


## 6. Write the small Gold exports

These are written to Databricks Volume storage first. They can then be copied into the repository's `data_sample/gold_exports/` folder as the student-created validated samples.

**Never manually edit a CSV value.**


In [0]:
# WEEK 08 — Cell 9: Export helper

EXPORT_BASE = "/Volumes/trippulse/default/trippulsedata/gold_exports/week08"

def write_single_csv(df, export_name):
    path = f"{EXPORT_BASE}/{export_name}"
    (
        df.coalesce(1)
          .write
          .mode("overwrite")
          .option("header", True)
          .csv(path)
    )
    return path


In [0]:
# WEEK 08 — Cell 10: Write all approved Gold export families

export_register = []

for name, df in exports.items():
    export_file = f"{name}.csv"
    export_path = write_single_csv(df, export_file)

    export_register.append(
        (
            export_file,
            f"{CATALOG}.{SCHEMA}.{name}",
            export_path,
            df.count(),
            len(df.columns),
            "deterministic Week 08 slice"
        )
    )

display(
    spark.createDataFrame(
        export_register,
        [
            "export_file",
            "source_gold_table",
            "export_path",
            "row_count_written",
            "column_count",
            "creation_rule"
        ]
    )
)

print(f"Approved export families written: {len(export_register)}")


export_file,source_gold_table,export_path,row_count_written,column_count,creation_rule
dim_date.csv,workspace.default.dim_date,/Volumes/trippulse/default/trippulsedata/gold_exports/week08/dim_date.csv,90,9,deterministic Week 08 slice
dim_zone.csv,workspace.default.dim_zone,/Volumes/trippulse/default/trippulsedata/gold_exports/week08/dim_zone.csv,120,6,deterministic Week 08 slice
dim_driver.csv,workspace.default.dim_driver,/Volumes/trippulse/default/trippulsedata/gold_exports/week08/dim_driver.csv,500,7,deterministic Week 08 slice
agg_trip_operations_daily.csv,workspace.default.agg_trip_operations_daily,/Volumes/trippulse/default/trippulsedata/gold_exports/week08/agg_trip_operations_daily.csv,56,12,deterministic Week 08 slice
agg_zone_demand_daily.csv,workspace.default.agg_zone_demand_daily,/Volumes/trippulse/default/trippulsedata/gold_exports/week08/agg_zone_demand_daily.csv,6678,8,deterministic Week 08 slice
agg_surge_impact_daily.csv,workspace.default.agg_surge_impact_daily,/Volumes/trippulse/default/trippulsedata/gold_exports/week08/agg_surge_impact_daily.csv,12476,10,deterministic Week 08 slice
agg_driver_performance_daily.csv,workspace.default.agg_driver_performance_daily,/Volumes/trippulse/default/trippulsedata/gold_exports/week08/agg_driver_performance_daily.csv,21954,9,deterministic Week 08 slice
agg_payment_reliability_daily.csv,workspace.default.agg_payment_reliability_daily,/Volumes/trippulse/default/trippulsedata/gold_exports/week08/agg_payment_reliability_daily.csv,36,9,deterministic Week 08 slice


Approved export families written: 8


In [0]:
# WEEK 08 — Cell 11: Read the written CSV back and prove row/column preservation

readback_results = []

for name, source_df in exports.items():
    export_path = f"{EXPORT_BASE}/{name}.csv"

    readback_df = (
        spark.read
             .option("header", True)
             .csv(export_path)
    )

    source_count = source_df.count()
    readback_count = readback_df.count()

    same_column_names = (
        source_df.columns == readback_df.columns
    )

    readback_results.append(
        (
            name,
            source_count,
            readback_count,
            source_count == readback_count,
            same_column_names,
            len(source_df.columns),
            len(readback_df.columns)
        )
    )

readback_df = spark.createDataFrame(
    readback_results,
    [
        "gold_table",
        "source_slice_rows",
        "export_rows",
        "row_count_pass",
        "column_name_pass",
        "source_columns",
        "export_columns"
    ]
)

display(readback_df)

if not all(
    r["row_count_pass"] and r["column_name_pass"]
    for r in readback_df.collect()
):
    raise ValueError(
        "Gold-to-export validation failed. Do not continue to Power BI."
    )


gold_table,source_slice_rows,export_rows,row_count_pass,column_name_pass,source_columns,export_columns
dim_date,90,90,true,true,9,9
dim_zone,120,120,true,true,6,6
dim_driver,500,500,true,true,7,7
agg_trip_operations_daily,56,56,true,true,12,12
agg_zone_demand_daily,6678,6678,true,true,8,8
agg_surge_impact_daily,12476,12476,true,true,10,10
agg_driver_performance_daily,21954,21954,true,true,9,9
agg_payment_reliability_daily,36,36,true,true,9,9


## 7. PBI-01 source traceability

The manual requires every first-page card/visual to be traceable to a Gold source and validation query.

Use this register directly when preparing `dashboard/README.md` and `docs/dashboard_insights.md`.


In [0]:
# WEEK 08 — Cell 12: PBI-01 source-to-report traceability register

trace_rows = [
    (
        "Total Trip Requests",
        "agg_trip_operations_daily",
        "trip_requests",
        "SUM(trip_requests)",
        "KPI card"
    ),
    (
        "Completion Rate",
        "agg_trip_operations_daily",
        "completed_trips, trip_requests",
        "DIVIDE(SUM(completed_trips), SUM(trip_requests))",
        "KPI card"
    ),
    (
        "Cancellation Rate",
        "agg_trip_operations_daily",
        "cancelled_trips, trip_requests",
        "DIVIDE(SUM(cancelled_trips), SUM(trip_requests))",
        "KPI card"
    ),
    (
        "Unfulfilled Rate",
        "agg_trip_operations_daily",
        "unfulfilled_trips, trip_requests",
        "DIVIDE(SUM(unfulfilled_trips), SUM(trip_requests))",
        "KPI card"
    ),
    (
        "Average Driver Response Minutes",
        "agg_trip_operations_daily",
        "avg_response_minutes",
        "Gold-approved response measure",
        "KPI card"
    ),
    (
        "Daily request/completion trend",
        "agg_trip_operations_daily",
        "operation_date, trip_requests, completed_trips",
        "Daily Gold fields",
        "Visual"
    ),
    (
        "Trip outcome by service",
        "agg_trip_operations_daily",
        "service_type, trip_requests, completed_trips, cancelled_trips, unfulfilled_trips",
        "Gold summary outcome fields",
        "Visual"
    ),
    (
        "Zone detail",
        "agg_zone_demand_daily",
        "operation_date, pickup_zone_id, service_type, trip_requests, completion/cancellation measures",
        "Gold summary fields",
        "Visual"
    )
]

display(
    spark.createDataFrame(
        trace_rows,
        [
            "report_item",
            "gold_source",
            "fields",
            "calculation_or_use",
            "pbi01_area"
        ]
    )
)


report_item,gold_source,fields,calculation_or_use,pbi01_area
Total Trip Requests,agg_trip_operations_daily,trip_requests,SUM(trip_requests),KPI card
Completion Rate,agg_trip_operations_daily,"completed_trips, trip_requests","DIVIDE(SUM(completed_trips), SUM(trip_requests))",KPI card
Cancellation Rate,agg_trip_operations_daily,"cancelled_trips, trip_requests","DIVIDE(SUM(cancelled_trips), SUM(trip_requests))",KPI card
Unfulfilled Rate,agg_trip_operations_daily,"unfulfilled_trips, trip_requests","DIVIDE(SUM(unfulfilled_trips), SUM(trip_requests))",KPI card
Average Driver Response Minutes,agg_trip_operations_daily,avg_response_minutes,Gold-approved response measure,KPI card
Daily request/completion trend,agg_trip_operations_daily,"operation_date, trip_requests, completed_trips",Daily Gold fields,Visual
Trip outcome by service,agg_trip_operations_daily,"service_type, trip_requests, completed_trips, cancelled_trips, unfulfilled_trips",Gold summary outcome fields,Visual
Zone detail,agg_zone_demand_daily,"operation_date, pickup_zone_id, service_type, trip_requests, completion/cancellation measures",Gold summary fields,Visual


## 8. Week 08 notebook final validation

This is the Databricks-side proof before moving into Power BI Desktop.


In [0]:
# WEEK 08 — Cell 13: Final notebook validation checklist

final_checks = [
    (
        "all_10_week07_gold_tables_exist",
        all(t in existing_tables for t in gold_tables)
    ),
    (
        "batch_only_week08_scope",
        True
    ),
    (
        "eight_approved_export_families_prepared",
        len(exports) == 8
    ),
    (
        "source_and_export_dataframe_columns_match",
        all(
            spark.table(name).columns == df.columns
            for name, df in exports.items()
        )
    ),
    (
        "source_and_export_dataframe_types_match",
        all(
            spark.table(name).schema == df.schema
            for name, df in exports.items()
        )
    ),
    (
        "csv_readback_row_counts_match",
        all(
            r["row_count_pass"]
            for r in readback_df.collect()
        )
    ),
    (
        "csv_readback_column_names_match",
        all(
            r["column_name_pass"]
            for r in readback_df.collect()
        )
    )
]

final_df = spark.createDataFrame(
    final_checks,
    ["check", "pass"]
)

display(final_df)

if not all(v for _, v in final_checks):
    raise ValueError(
        "WEEK 08 NOTEBOOK VALIDATION = FAIL. "
        "Fix the failed check before building PBI-01."
    )

print("WEEK 08 NOTEBOOK VALIDATION = PASS")


check,pass
all_10_week07_gold_tables_exist,true
batch_only_week08_scope,true
eight_approved_export_families_prepared,true
source_and_export_dataframe_columns_match,true
source_and_export_dataframe_types_match,true
csv_readback_row_counts_match,true
csv_readback_column_names_match,true


WEEK 08 NOTEBOOK VALIDATION = PASS


# 9. Power BI work after this notebook

Create/update the required repository artifacts:

- `data_sample/gold_exports/` — small validated student-created Gold exports
- `dashboard/powerbi_dashboard.pbix`
- `dashboard/README.md`
- `docs/dashboard_insights.md`
- `screenshots/`
- `weekly_logs/week08_log.md`

## PBI-01 — Ride Operations Overview

**Decision question:**  
Are trusted ride requests being fulfilled efficiently, and where are the main operational losses?

### Five KPI cards
1. Total Trip Requests
2. Completion Rate
3. Cancellation Rate
4. Unfulfilled Rate
5. Average Driver Response Minutes

### Approved visuals
- Daily request/completion trend
- Trip status by service
- Duration/fare trend
- Top pickup zones or approved secondary/detail view

### Approved filters
- Date
- `service_type`
- pickup `zone_type`

### Approved PBI-01 Gold sources
- `agg_trip_operations_daily`
- `agg_zone_demand_daily`
- `dim_date`
- `dim_zone`

**Do not connect Power BI to raw/source, Bronze, Silver Candidate, Trusted Silver detail, quarantine or technical-validation reference outputs.**


# Week 08 exit evidence

Before closing Week 08, capture:

1. Gold-to-export row/column/type/grain proof.
2. Power BI Model view showing approved one-to-many dimension relationships and no ambiguous fact-to-fact path.
3. PBI-01 page screenshot with the approved five cards, visuals and filters.
4. At least two PBI-01 measures reconciled to the same filtered Gold slice.
5. `dashboard/README.md` with source map, relationships, refresh/open steps and limitations.
6. `docs/dashboard_insights.md` with the initial evidence-backed observation.
7. `weekly_logs/week08_log.md` with work, checks, ownership, blockers/rework and AI transparency.

**Week 08 is complete only when both the Databricks hand-off evidence and Power BI evidence pass.**


## Gold → Power BI measure reconciliation

## PBI - 01 : Validate Total Trip Requests + Completion Rate from 01-01-26 to 14-01-26

In [0]:
from pyspark.sql import functions as F

gold_check = (
    spark.table("workspace.default.agg_trip_operations_daily")
    .filter(
        (F.col("operation_date") >= F.to_date(F.lit("2026-01-01"))) &
        (F.col("operation_date") <= F.to_date(F.lit("2026-01-14")))
    )
    .agg(
        F.sum("trip_requests").alias("gold_total_trip_requests"),
        F.sum("completed_trips").alias("gold_completed_trips")
    )
    .withColumn(
        "gold_completion_rate",
        F.when(
            F.col("gold_total_trip_requests") != 0,
            F.col("gold_completed_trips") / F.col("gold_total_trip_requests")
        ).otherwise(F.lit(None))
    )
)

display(gold_check)

gold_total_trip_requests,gold_completed_trips,gold_completion_rate
37892,24297,0.6412171434603611


## PBI - 01 : Validate Cancellation Rate from 01-01-26 to 14-01-26

In [0]:
cancellation_check = (
    spark.table("workspace.default.agg_trip_operations_daily")
    .filter(
        (F.col("operation_date") >= F.to_date(F.lit("2026-01-01"))) &
        (F.col("operation_date") <= F.to_date(F.lit("2026-01-14")))
    )
    .agg(
        F.sum("trip_requests").alias("gold_total_trip_requests"),
        F.sum("cancelled_trips").alias("gold_cancelled_trips")
    )
    .withColumn(
        "gold_cancellation_rate",
        F.when(
            F.col("gold_total_trip_requests") != 0,
            F.col("gold_cancelled_trips") / F.col("gold_total_trip_requests")
        ).otherwise(F.lit(None))
    )
)

display(cancellation_check)

gold_total_trip_requests,gold_cancelled_trips,gold_cancellation_rate
37892,9027,0.23822970547872901


## PBI-02 — Gold Reconciliation: Trip Requests, Completion Rate and Cancellation Rate

In [0]:
%sql
SELECT
    SUM(trip_requests) AS total_trip_requests,
    SUM(completed_trips) / NULLIF(SUM(trip_requests), 0) AS completion_rate,
    SUM(cancellation_rate * trip_requests) / NULLIF(SUM(trip_requests), 0) AS cancellation_rate
FROM workspace.default.agg_zone_demand_daily
WHERE operation_date BETWEEN '2026-01-01' AND '2026-01-14';

total_trip_requests,completion_rate,cancellation_rate
37892,0.6412171434603611,0.238230


## PBI-02 — Gold Reconciliation: Surge Trip Share and Average Final Fare

In [0]:
%sql
SELECT
    SUM(surge_trip_count) / NULLIF(SUM(trip_requests), 0) AS surge_trip_share,

    SUM(
        avg_final_fare_inr * trip_requests * completion_rate
    )
    /
    NULLIF(
        SUM(trip_requests * completion_rate),
        0
    ) AS average_final_fare
FROM workspace.default.agg_surge_impact_daily
WHERE operation_date BETWEEN '2026-01-01' AND '2026-01-14';

surge_trip_share,average_final_fare
0.5528079805763749,446.614775


## PBI-03 — Driver Slice Gold Reconciliation

In [0]:
%sql
SELECT
    SUM(completed_trips) / NULLIF(SUM(assigned_requests), 0) AS driver_reliability_rate,

    SUM(avg_response_minutes * assigned_requests)
        / NULLIF(SUM(assigned_requests), 0) AS average_driver_response_minutes,

    SUM(avg_trip_duration_minutes * completed_trips)
        / NULLIF(SUM(completed_trips), 0) AS average_trip_duration
FROM workspace.default.agg_driver_performance_daily
WHERE operation_date BETWEEN '2026-01-01' AND '2026-01-14'
  AND driver_id = 'DRV-000008';

driver_reliability_rate,average_driver_response_minutes,average_trip_duration
0.8888888888888888,3.762962962962963,35.81875


## PBI-03 — Payment Method Slice Gold Reconciliation

In [0]:
%sql
SELECT
    SUM(successful_attempts)
        / NULLIF(SUM(payment_attempts), 0) AS payment_attempt_success_rate,

    AVG(avg_attempts_per_trip) AS average_attempts_per_trip
FROM workspace.default.agg_payment_reliability_daily
WHERE payment_date BETWEEN '2026-01-01' AND '2026-01-14'
  AND payment_method = 'upi';

payment_attempt_success_rate,average_attempts_per_trip
0.36299607205742923,1.40505068643057813571
